# 01 - Build and freeze a RAG corpus

Notebook này chỉ làm **Ingestion**. Nó nhận file, folder Kaggle Dataset hoặc ZIP; sau khi hoàn tất sẽ
đóng băng kết quả thành `corpus_bundle.zip`. Notebook không load answer model hay mở chat.

Luồng xử lý (chi tiết: `docs/data-flow.md`):

1. Parse + PaddleOCR-VL (GPU 1). Text OCR được LLM sửa chính tả song song trên GPU 0, mỗi lần chỉ nhìn
   system + skill + đoạn liền trước đã sửa + đoạn hiện tại.
2. VLM (tuỳ chọn, chart/flowchart) chạy sau khi sửa OCR, nhận OCR đã sửa + đúng 1 trang liền trước.
3. Hồ sơ tài liệu: tiêu đề, số hiệu, ngày, người ký, keyword, thực thể.
4. Chunking theo cấu trúc: 1 chunk = nhóm đoạn / 1 bảng / 1 ảnh; bảng lớn = preview + file `.xlsx`.
5. Embedding + Qdrant + BM25, rồi đóng băng corpus.


## 1. Cài đặt và import source

In [ ]:
from datetime import datetime as _CellDateTime
import time as _cell_time

_cell_label = '01_ingestion / 1. Cài đặt và import source'
_cell_started_at = _CellDateTime.now().astimezone()
_cell_started_perf = _cell_time.perf_counter()
print(f"[CELL START] {_cell_label} | {_cell_started_at.isoformat(timespec='seconds')}")
try:
    pass
    import os
    import subprocess
    import sys
    from pathlib import Path

    REPO_URL = "https://github.com/NhoDung/RAG-basic.git"
    REPO_DIR = Path("/kaggle/working/RAG-basic")

    if not (Path.cwd() / "rag_kaggle").exists():
        if REPO_DIR.exists():
            subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
        else:
            subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
        os.chdir(REPO_DIR)

    PROJECT_DIR = Path.cwd()
    sys.path.insert(0, str(PROJECT_DIR))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"])
    print("Project:", PROJECT_DIR)
finally:
    _cell_finished_at = _CellDateTime.now().astimezone()
    _cell_elapsed_seconds = _cell_time.perf_counter() - _cell_started_perf
    print(f"[CELL END] {_cell_label} | {_cell_finished_at.isoformat(timespec='seconds')}")
    print(f"[CELL ELAPSED_SECONDS] {_cell_label} | {_cell_elapsed_seconds:.3f}")


## 2. Chọn model trực tiếp và khai báo input

In [ ]:
from datetime import datetime as _CellDateTime
import time as _cell_time

_cell_label = '01_ingestion / 2. Chọn model và khai báo input'
_cell_started_at = _CellDateTime.now().astimezone()
_cell_started_perf = _cell_time.perf_counter()
print(f"[CELL START] {_cell_label} | {_cell_started_at.isoformat(timespec='seconds')}")
try:
    pass
    # Mỗi phần nhận trực tiếp Hugging Face model ID hoặc tên PaddleOCR.
    OCR_MODEL = "PaddleOCR-VL-1.6"
    OCR_CORRECTION_MODEL = "Qwen/Qwen2.5-3B-Instruct"  # None disables OCR spelling correction.
    # Glossary tên riêng xuyên suốt document cho bước sửa OCR; bật sau khi đã benchmark với model sửa lỗi.
    OCR_CORRECTION_GLOSSARY = False
    OCR_CORRECTION_PREVIOUS_CHARS = 600  # đuôi đoạn liền trước (đã sửa) đưa vào prompt
    OCR_CORRECTION_SEGMENT_CHARS = 1200  # đoạn dài được sửa theo từng khúc <= giá trị này
    OCR_CORRECTION_TABLES = False  # True: sửa bảng theo từng ô, giữ nguyên số hàng/ô
    # VLM chạy sau khi sửa OCR của cả document (nhận OCR đã sửa + 1 trang liền trước), nên có thể bật cùng
    # OCR_CORRECTION_MODEL. Chỉ áp dụng cho chart/flowchart.
    VISION_MODEL = None  # Ví dụ: "Qwen/Qwen2.5-VL-3B-Instruct"
    EMBEDDING_MODEL = "BAAI/bge-m3"
    EMBEDDING_REVISION = None
    EMBEDDING_QUERY_INSTRUCTION = None

    # Chunking theo cấu trúc: bảng nhỏ giữ nguyên dạng markdown; bảng lớn chỉ index preview
    # và lưu đầy đủ trong file .xlsx (đi kèm corpus_bundle.zip).
    TEXT_CHUNK_CHARS = 1800
    TABLE_INLINE_MAX_CHARS = 6000
    TABLE_PREVIEW_ROWS = 5
    TABLE_PREVIEW_COLUMNS = 5

    # Liệt kê tên các dataset theo đường dẫn thực tế được Kaggle mount.
    INPUT_SOURCES = [
        "/kaggle/input/datasets/huythinhuet/raw-data-small",
    ]
    # Chuẩn hóa ngay tại đây để các cell phía sau có thể gọi .exists(), .name, v.v.
    INPUT_SOURCES = [Path(source) for source in INPUT_SOURCES]

    # Có thể thêm đường dẫn file/folder cụ thể nếu cần:
    # INPUT_SOURCES.append(Path("/kaggle/input/datasets/owner/another-dataset/report.pdf"))

    WORK_DIR = Path("/kaggle/working/rag_ingestion")
    OUTPUT_BUNDLE = Path("/kaggle/working/corpus_bundle.zip")
    INCLUDE_SOURCE_DOCUMENTS = False
finally:
    _cell_finished_at = _CellDateTime.now().astimezone()
    _cell_elapsed_seconds = _cell_time.perf_counter() - _cell_started_perf
    print(f"[CELL END] {_cell_label} | {_cell_finished_at.isoformat(timespec='seconds')}")
    print(f"[CELL ELAPSED_SECONDS] {_cell_label} | {_cell_elapsed_seconds:.3f}")


## 3. Cài PaddleOCR-VL trong môi trường riêng

In [ ]:
from datetime import datetime as _CellDateTime
import time as _cell_time

_cell_label = '01_ingestion / 3. Cài PaddleOCR-VL'
_cell_started_at = _CellDateTime.now().astimezone()
_cell_started_perf = _cell_time.perf_counter()
print(f"[CELL START] {_cell_label} | {_cell_started_at.isoformat(timespec='seconds')}")
try:
    pass
    import shutil

    INSTALL_OCR = OCR_MODEL is not None
    PADDLE_VERSION = "3.2.1"
    PADDLE_CUDA = "cu126"
    OCR_ENV = Path("/tmp/paddle_env")
    OCR_PYTHON = None


    def run(command):
        print("$", " ".join(map(str, command)))
        completed = subprocess.run(list(map(str, command)), capture_output=True, text=True)
        if completed.returncode != 0:
            print(completed.stdout[-2000:])
            print(completed.stderr[-4000:])
            raise RuntimeError(f"Command failed ({completed.returncode}): {command[0]}")
        return completed.stdout


    def python_works(path):
        return path.exists() and subprocess.run(
            [str(path), "-m", "pip", "--version"],
            capture_output=True,
            text=True,
        ).returncode == 0


    if INSTALL_OCR:
        try:
            python = OCR_ENV / "bin" / "python"
            if not python_works(python):
                shutil.rmtree(OCR_ENV, ignore_errors=True)
                try:
                    # Một số Kaggle image không có ensurepip đầy đủ.
                    run([sys.executable, "-m", "venv", str(OCR_ENV)])
                except Exception:
                    shutil.rmtree(OCR_ENV, ignore_errors=True)
                    print("python -m venv failed; retrying with virtualenv...")
                    run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
                    run([sys.executable, "-m", "virtualenv", str(OCR_ENV)])
            if not python_works(python):
                raise RuntimeError(f"OCR environment was created but Python is unusable: {python}")
            pip = [str(python), "-m", "pip", "install", "-q"]
            run(pip + ["--upgrade", "pip"])
            run(pip + [
                f"paddlepaddle-gpu=={PADDLE_VERSION}",
                "-i", f"https://www.paddlepaddle.org.cn/packages/stable/{PADDLE_CUDA}/",
                "--extra-index-url", "https://pypi.org/simple",
            ])
            run(pip + ["paddleocr[doc-parser]"])
            print(run([
                python,
                "-c",
                "import paddle, paddleocr; "
                "print('paddle', paddle.__version__, 'cuda', paddle.device.is_compiled_with_cuda()); "
                "print('paddleocr', paddleocr.__version__)",
            ]))
            OCR_PYTHON = str(python)
        except Exception as exc:
            print("PaddleOCR-VL setup failed; OCR will be disabled:", exc)

    print("OCR_PYTHON =", OCR_PYTHON)
finally:
    _cell_finished_at = _CellDateTime.now().astimezone()
    _cell_elapsed_seconds = _cell_time.perf_counter() - _cell_started_perf
    print(f"[CELL END] {_cell_label} | {_cell_finished_at.isoformat(timespec='seconds')}")
    print(f"[CELL ELAPSED_SECONDS] {_cell_label} | {_cell_elapsed_seconds:.3f}")


## 4. Tạo ingestion runtime và kiểm tra tài nguyên

In [ ]:
from datetime import datetime as _CellDateTime
import time as _cell_time

_cell_label = '01_ingestion / 4. Tạo ingestion runtime'
_cell_started_at = _CellDateTime.now().astimezone()
_cell_started_perf = _cell_time.perf_counter()
print(f"[CELL START] {_cell_label} | {_cell_started_at.isoformat(timespec='seconds')}")
try:
    pass
    from rag_kaggle import (
        IngestionPipeline, PipelineConfig, configure_ingestion_devices,
        inspect_resources, suggest_model_upgrades,
    )

    config = PipelineConfig.from_yaml(PROJECT_DIR / "configs" / "baseline.yaml")
    config.work_dir = WORK_DIR
    config.parsing.ocr_model_name = OCR_MODEL or "PaddleOCR-VL-1.6"
    config.parsing.ocr_python = OCR_PYTHON
    config.parsing.enable_ocr = OCR_MODEL is not None and OCR_PYTHON is not None
    config.ocr_correction.enabled = OCR_CORRECTION_MODEL is not None
    if OCR_CORRECTION_MODEL:
        config.ocr_correction.model = OCR_CORRECTION_MODEL
    config.ocr_correction.glossary_enabled = OCR_CORRECTION_GLOSSARY
    config.ocr_correction.previous_context_chars = OCR_CORRECTION_PREVIOUS_CHARS
    config.ocr_correction.segment_chars = OCR_CORRECTION_SEGMENT_CHARS
    config.ocr_correction.correct_tables = OCR_CORRECTION_TABLES
    config.vision.enabled = VISION_MODEL is not None
    if VISION_MODEL:
        config.vision.model = VISION_MODEL
    config.chunking.text_chunk_chars = TEXT_CHUNK_CHARS
    config.chunking.table_inline_max_chars = TABLE_INLINE_MAX_CHARS
    config.chunking.table_preview_rows = TABLE_PREVIEW_ROWS
    config.chunking.table_preview_columns = TABLE_PREVIEW_COLUMNS
    config.retrieval.dense_model = EMBEDDING_MODEL
    config.retrieval.dense_fallback_model = EMBEDDING_MODEL  # Không âm thầm đổi sang model khác.
    config.retrieval.dense_revision = EMBEDDING_REVISION
    config.retrieval.dense_query_instruction = EMBEDDING_QUERY_INSTRUCTION
    config.artifacts.include_source_documents = INCLUDE_SOURCE_DOCUMENTS

    resources = inspect_resources()
    print("Resources:", resources)
    print("Allocation:", configure_ingestion_devices(config))
    print("Suggestions:", suggest_model_upgrades("ingestion", resources, config))
    pipeline = IngestionPipeline(config)
finally:
    _cell_finished_at = _CellDateTime.now().astimezone()
    _cell_elapsed_seconds = _cell_time.perf_counter() - _cell_started_perf
    print(f"[CELL END] {_cell_label} | {_cell_finished_at.isoformat(timespec='seconds')}")
    print(f"[CELL ELAPSED_SECONDS] {_cell_label} | {_cell_elapsed_seconds:.3f}")


## 5. Ingest, freeze và export

In [ ]:
from datetime import datetime as _CellDateTime
import time as _cell_time

_cell_label = '01_ingestion / 5. Ingest, freeze và export'
_cell_started_at = _CellDateTime.now().astimezone()
_cell_started_perf = _cell_time.perf_counter()
print(f"[CELL START] {_cell_label} | {_cell_started_at.isoformat(timespec='seconds')}")
try:
    pass
    from rag_kaggle.ingestion import discover_input_files

    print("Configured input sources:")
    for source in INPUT_SOURCES:
        source_path = Path(source)
        print(" -", source_path, "exists=" + str(source_path.exists()))

    files, discovery = discover_input_files(INPUT_SOURCES, config)
    print(f"Found {len(files)} supported document(s):")
    for path in files:
        print(" -", path)
    if discovery["skipped"]:
        print("Skipped/not found:")
        for item in discovery["skipped"][:30]:
            print(" -", item)
    if not files:
        kaggle_input_root = Path("/kaggle/input/datasets")
        mounted = sorted(str(path) for path in kaggle_input_root.glob("*/*")) if kaggle_input_root.exists() else []
        raise ValueError(
            f"Không tìm thấy tài liệu hỗ trợ. Dataset đang được mount: {mounted}. "
            f"Hãy sửa INPUT_SOURCES theo đúng đường dẫn hiển thị trên Kaggle."
        )

    report = pipeline.ingest(files, reset=True, progress=print)
    report["input_discovery"] = discovery
    print("Ingestion stats:", report["stats"])
    for item in report["documents"]:
        profile = item.get("profile") or {}
        print(
            f" - {item['file']}: {item['chunks']} chunks, {item['entities']} entities | "
            f"title={profile.get('title')!r} | số hiệu={profile.get('doc_number')} | "
            f"ngày={profile.get('issue_date')} | người ký={profile.get('signers')}"
        )
        print("   keywords:", (profile.get("keywords") or [])[:8])
        correction = item.get("ocr_correction")
        if correction:
            print(
                f"   OCR correction: {correction.get('success_count', 0)} ok, "
                f"{correction.get('fallback_count', 0)} fallback, {correction.get('skipped_count', 0)} skipped, "
                f"{correction.get('segment_count', 0)} segments, buffer peak {correction.get('buffer_peak_size', 0)}"
            )
            if correction.get("name_variants"):
                print("   name variants seen:", correction["name_variants"])

    from collections import Counter

    chunks = pipeline.metadata.list_chunks()
    print("Chunks by type:", dict(Counter(chunk.chunk_type for chunk in chunks)))
    truncated = [chunk for chunk in chunks if chunk.metadata.get("table_truncated")]
    tables = sorted((config.work_dir / "tables").rglob("*.xlsx"))
    print(f"Large tables indexed as preview: {len(truncated)} | .xlsx files written: {len(tables)}")
    for chunk in truncated[:5]:
        print(" -", chunk.source_file, chunk.metadata["table_rows"], "rows ->", chunk.metadata["table_file"])
    if not report["ok"]:
        raise RuntimeError(report["errors"])

    bundle = pipeline.export_corpus_bundle(OUTPUT_BUNDLE)
    print("Frozen corpus:", pipeline.validate_corpus())
    print("Bundle:", bundle, f"({bundle.stat().st_size / 1024**2:.1f} MB)")

    from IPython.display import FileLink, display
    display(FileLink(str(bundle)))
finally:
    _cell_finished_at = _CellDateTime.now().astimezone()
    _cell_elapsed_seconds = _cell_time.perf_counter() - _cell_started_perf
    print(f"[CELL END] {_cell_label} | {_cell_finished_at.isoformat(timespec='seconds')}")
    print(f"[CELL ELAPSED_SECONDS] {_cell_label} | {_cell_elapsed_seconds:.3f}")
